# Clase 11. Arquitecturas de agentes: de A3C a IMPALA

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

Bienvenida o bienvenido a la clase 11. En la clase anterior viste cómo Rainbow combina varias mejoras sobre la Red Q Profunda (DQN, del inglés Deep Q-Network) y cómo la perspectiva distribucional cambia la forma de estimar el valor de una acción. Todo eso responde a una pregunta: ¿cómo hacer que un solo agente aprenda mejor? Hoy respondes a una pregunta distinta y, para entornos complejos a gran escala, igual de importante: ¿cómo hacer que muchos agentes aprendan más rápido, juntos?

Piensa en los casos de uso que ves en la clase 12, la última del curso: sistemas capaces de jugar Go, ajedrez o StarCraft a un nivel sobrehumano. Ninguno se entrenó con un único agente en una única máquina durante semanas. Se entrenaron con cientos o miles de copias del agente jugando en paralelo, acumulando experiencia a una velocidad que ninguna máquina aislada podría igualar. Hoy aprendes esa arquitectura: primero el Actor-Crítico de Ventaja Asíncrono (A3C, del inglés Asynchronous Advantage Actor-Critic), que sustituye la memoria de repetición por el paralelismo, y después IMPALA, que lleva esa misma idea de un ordenador a un clúster completo.

## Guion de la clase

1. El cuello de botella que deja atrás la Red Q Profunda
2. A3C: sustituye la memoria por el paralelismo
3. Consejos prácticos de ingeniería
4. IMPALA: de un ordenador a un clúster, con V-trace
5. Código en vivo: "Catch" con un actor-crítico simple
6. Puntos clave y qué viene en la clase 12


## 1. El cuello de botella que deja atrás la Red Q Profunda

*Guion de clase: antes de escribir nada en la pizarra, pregunta a la clase qué dos piezas incorporó DQN para poder aprender de forma estable (memoria de repetición y red objetivo) y por qué hacían falta. Deja que respondan antes de seguir.*

Recuerda los dos problemas de correlación que obligaron a DQN a incorporar una memoria de repetición y una red objetivo:

- Los datos que genera un agente jugando de forma continua están muy correlacionados entre pasos consecutivos
- Una red neuronal, por su tendencia natural a producir funciones suaves, asigna valores parecidos a estados parecidos, lo que hace que el objetivo de entrenamiento se persiga a sí mismo

La memoria de repetición resuelve ambos problemas guardando hasta un millón de fotogramas y muestreando lotes aleatorios. Pero a cambio introduce una dependencia estricta del aprendizaje fuera de política y deja sin aprovechar buena parte de la capacidad de una unidad de procesamiento gráfico (GPU): las redes pequeñas que funcionan bien en este esquema no llegan a saturar su rendimiento, así que entrenar un solo juego de Atari podía llevar hasta una semana en una sola GPU.

**La pregunta que abrió la puerta a A3C fue directa: si el problema de fondo es la correlación de los datos, ¿hace falta almacenar experiencia pasada para romperla, o basta con generar experiencia diversa en el momento?**


## 2. A3C: sustituye la memoria por el paralelismo

*Guion de clase: cuenta la analogía en voz alta, sin mostrar todavía ningún esquema. Es la pieza que más ayuda a que la idea "haga clic".*

Imagina que, en vez de una sola persona jugando una partida tras otra y anotando cada jugada en un cuaderno para repasarlo más tarde, pones a 16 personas jugando partidas distintas al mismo tiempo, cada una en su propio tablero. Cada cierto tiempo, cada jugadora consulta la estrategia común más reciente, juega unos cuantos turnos con ella, calcula qué ajuste haría mejor esa estrategia y lo susurra a un cuaderno compartido, sin esperar turno ni pedir permiso a las demás. Como todas juegan situaciones distintas a la vez, ningún ajuste individual arrastra al cuaderno común hacia un rincón sesgado: la diversidad simultánea entre jugadoras cumple el mismo papel que antes cumplía la diversidad temporal de la memoria de repetición, solo que sin guardar nada.

Así funciona A3C. Cada trabajador —un hilo de proceso con su propia copia del entorno— repite este ciclo:

1. Toma los parámetros más recientes de la red compartida
2. Juega un número fijo de pasos con la política actual
3. Calcula un gradiente y lo aplica de forma asíncrona a esa red compartida, sin bloquear a los demás trabajadores ni sincronizarse con ellos

No hay memoria de repetición ni red objetivo: la propia concurrencia de trabajadores explorando puntos distintos del entorno estabiliza el aprendizaje.

### El algoritmo dentro de cada trabajador: A2C

El componente que aprende dentro de cada trabajador es un Actor-Crítico de Ventaja (A2C, del inglés Advantage Actor-Critic), la misma familia que trabajaste en la clase nueve: una red que predice, a la vez, una distribución de probabilidad sobre acciones (el actor) y una estimación del valor del estado (el crítico).

El actor trunca el retorno a los n pasos siguientes, usa el crítico para completar el resto por *bootstrapping* y resta el valor del estado actual como referencia; a esa diferencia se le llama **ventaja**, y es lo que multiplica al gradiente de la política. Hay una razón adicional por la que este esquema es más estable que ajustar valores Q sueltos: la salida del actor es una distribución que suma uno, así que aumentar la probabilidad de una acción reduce automáticamente la de las demás. Es una forma de estabilización numérica que viene incorporada en el propio diseño, algo que un valor Q sin restricciones no tiene.

### Lo que muestran los números

*Guion de clase: pide a la clase que prediga, antes de mostrar la cifra, si A3C escalará mejor o peor que las variantes de un solo paso al pasar de un hilo a 16.*

Los experimentos de escalado con 16 núcleos de una unidad central de procesamiento (CPU) y 16 trabajadores frente a DQN con GPU, en siete juegos de Atari, dejan cifras concretas:

- El actor-crítico escaló de forma sublineal: entre 12 y 13 veces más rápido con 16 hilos, frente a las 16 veces ideales
- Las variantes de un solo paso escalaron de forma superlineal, unas 20 veces más rápido, gracias al efecto estabilizador de tener varios trabajadores explorando a la vez
- En Breakout, un solo trabajador con 50 millones de fotogramas llegaba a unos 250 puntos; los mismos 50 millones de fotogramas repartidos entre 16 trabajadores llegaban a unos 300 puntos, por el efecto de promediar sobre una porción mayor del espacio de estados
- Pese a que las variantes de un paso escalaban mejor en velocidad bruta, A3C seguía siendo, en conjunto, la opción más eficiente en datos

El resultado práctico: A3C igualaba o superaba a DQN en Atari en un solo día, en una sola máquina, sin ninguna GPU; con cuatro días de entrenamiento, mejoraba todavía más. Funciona tanto con redes recurrentes como con redes hacia delante, fue uno de los primeros métodos capaces de aprender a navegar laberintos tridimensionales aleatorios a partir de píxeles, y se adapta a control continuo sustituyendo la distribución sobre acciones discretas por la media y la varianza de una distribución gaussiana.

**Tiene también límites explícitos**: puede ser menos estable que DQN, a veces necesita ajustar hiperparámetros juego a juego, aprovecha peor la GPU y, sobre todo, sigue atado a los hilos de una única máquina. Ese último límite es justo el que resuelve IMPALA.


## 3. Consejos prácticos de ingeniería

*Guion de clase: presenta esta sección como el puente hacia el código en vivo. Insiste en que "depura primero en pequeño" no es una sugerencia opcional, es el criterio que vas a aplicar en un momento con "Catch".*

Antes de escalar a algo como A3C, conviene interiorizar un principio de trabajo cueste lo que cueste: depura primero en pequeño.

- **Empieza con un problema de juguete**: elige un entorno mínimo que comparta las características clave del problema real (asignación de crédito a largo plazo, tamaño del espacio de entrada) pero que puedas resolver en minutos en tu portátil. El ejemplo recomendado es "Catch": una pelota cae desde arriba de una cuadrícula pequeña y controlas una pala que puedes mover a izquierda o derecha; recibes más uno si la atrapas y menos uno si falla. Puedes subir la dificultad poco a poco para encontrar exactamente el punto donde tu algoritmo empieza a fallar
- **Visualízalo todo**: curvas de entrenamiento, comportamiento de la política, predicciones del crítico. Una tendencia que no se ve en una tabla de números salta a la vista en una gráfica
- **Desconfía del diseño de recompensas**: aunque en teoría la recompensa forma parte de la definición del problema, en la práctica se ajusta constantemente. El riesgo real está en sobreespecificar recompensas intermedias: en una tarea robótica de alcanzar, mover y soltar un objeto, premiar demasiado los pasos intermedios puede enseñar al robot a alcanzar y mover el objeto, pero nunca a soltarlo, porque sostenerlo ya le resulta la política más estable y mejor recompensada
- **Trata el factor de descuento como un hiperparámetro más**: aunque el objetivo final no dependa de él en teoría, descontar reduce la varianza de tus estimaciones de retorno y puede encajar mejor con el sesgo inductivo de tu optimizador
- **Ajusta el tamaño de red con cuidado**: empieza pequeño porque entrena más rápido, pero prueba redes más grandes de vez en cuando. Con redes más grandes, la red objetivo resulta menos crítica, porque una red grande confunde menos estados parecidos entre sí
- **No inicialices la política de forma determinista**: desde el primer paso, debe muestrear acciones con cierta probabilidad, no repetir siempre la misma
- **Mantente al día con el aprendizaje supervisado**: RMSprop y Adam son los optimizadores de referencia en RL profundo; técnicas como el *dropout* merecen probarse, aunque la normalización por lotes todavía no está muy extendida en este campo

**¡Aplica ahora mismo el primer consejo!** El código que vas a ejecutar en un momento construye exactamente ese problema de juguete: "Catch".


## 4. IMPALA: de un ordenador a un clúster, con V-trace

*Guion de clase: dibuja en la pizarra dos máquinas mandando gradientes a un servidor central, con flechas de tiempos distintos, para que se vea físicamente por qué un gradiente puede llegar "viejo".*

A3C resuelve el problema dentro de una sola máquina, pero su arquitectura no crece bien más allá de eso: si simplemente repartes varios aprendices —cada uno con su propia copia de los pesos— entre distintas máquinas y les pides que sincronicen gradientes contra un servidor central, aparece un problema nuevo. Para cuando el gradiente de una máquina llega al servidor, otras máquinas ya lo han actualizado, así que ese gradiente llega desactualizado y termina empujando la red en una dirección que ya no corresponde al estado actual de los pesos.

### La solución: centraliza el aprendizaje, distribuye la actuación

La solución de IMPALA (del inglés Importance Weighted Actor-Learner Architecture) es cambiar qué se distribuye. En vez de repartir aprendices, centraliza el aprendizaje en una sola copia —normalmente sobre GPU— y distribuye solo la parte de actuar: decenas o cientos de actores en CPU recorren el entorno con la política que tienen a mano y envían su experiencia al aprendiz central.

Esto evita el problema del gradiente obsoleto, pero deja un problema hermano: para cuando la experiencia de un actor llega al aprendiz, los pesos de este ya pueden haber cambiado, así que esa experiencia fue generada por una política ligeramente distinta de la actual.

### V-trace: la corrección que da nombre a la arquitectura

Aquí entra V-trace, el método de corrección fuera de política. Usa muestreo por importancia para reponderar cada actualización según la diferencia entre la política de comportamiento μ del actor que generó los datos y la política objetivo π del aprendiz en el momento de usarlos.

- Si ambas coinciden, la actualización de V-trace se reduce exactamente a la actualización de A3C
- Cuanto más se separan, más peso pierde esa experiencia en el entrenamiento

Es un compromiso explícito entre sesgo y varianza: V-trace introduce algo de sesgo a cambio de reducir la varianza y mantener el entrenamiento estable, incluso con datos ligeramente desactualizados.

### Las cifras: velocidad y, sobre todo, transferencia

*Guion de clase: separa claramente las dos familias de cifras. La de velocidad es la que la clase espera; la de transferencia entre tareas es la que de verdad justifica el rediseño.*

En entornos tridimensionales, la velocidad de generación de datos escala así:

- A3C en una sola máquina, con 64 CPU y 32 hilos: entre 6.000 y 9.000 fotogramas por segundo
- A3C distribuido con 200 trabajadores: hasta unos 50.000 fotogramas por segundo
- IMPALA: entre 200.000 y 250.000 fotogramas por segundo

Pero la cifra más interesante no es de velocidad, sino de transferencia entre tareas. En el conjunto de 30 tareas cognitivas de DeepMind Lab (DMLab-30), entrenar una red por separado en cada tarea y entrenarla en las 30 a la vez son experimentos distintos:

- Con A3C y una red profunda, el rendimiento medio ronda el 30% del nivel humano
- IMPALA con una red pequeña ya supera esa cifra
- IMPALA con una red profunda entrenada juego a juego mejora todavía más
- El mejor resultado de todos es IMPALA con una red residual profunda entrenada en las 30 tareas simultáneamente con una única red, por encima incluso de la versión entrenada juego a juego por separado

Esa es la señal de **transferencia positiva**: aprender varias tareas a la vez, con la arquitectura adecuada, hace que cada tarea individual se aprenda mejor y con menos datos, no peor. En tiempo de entrenamiento, IMPALA con red profunda alcanza el nivel de A3C en menos de un día; A3C necesita alrededor de una semana para llegar ahí.

En el conjunto de 57 juegos de Atari, la distancia se repite: A3C distribuido entrenado por separado en cada juego llega a un 54% de puntuación media con red pequeña y a un 118% con red profunda; IMPALA supera a A3C en ambos tamaños de red, y con una red residual profunda entrenada por separado se acerca al 200% de puntuación media. El dato más llamativo, sin embargo, es otro: **IMPALA entrenado en los 57 juegos a la vez, con una sola red, alcanza en torno al 60% de puntuación media** —comparable a la DQN original entrenada juego por juego— pese a que la diversidad visual y de mecánicas entre 57 juegos de Atari es mucho mayor que entre las 30 tareas de DeepMind Lab.

### Una pregunta abierta

Si IMPALA es más rápido y transfiere mejor entre tareas, ¿por qué DQN sigue siendo, en algunos aspectos, más estable que A3C? La respuesta apunta al entrenamiento por lotes: muestrear un lote aleatorio de la memoria de repetición es, en sí mismo, un ingrediente estabilizador importante, y hasta pequeñas desviaciones respecto a la política actual afectan al actor-crítico. V-trace compensa buena parte de ese efecto, pero si la experiencia llega demasiado desactualizada, los pesos de importancia se acercan a cero o quedan recortados en uno, y esa experiencia deja de aportar señal de aprendizaje útil.


## 5. Código en vivo: "Catch" con un actor-crítico simple

*Guion de clase: antes de tocar una sola línea de código, di esto en voz alta: "lo que vamos a construir ahora NO es A3C ni IMPALA". A3C necesita varios procesos jugando en paralelo y actualizando una red compartida de forma asíncrona; IMPALA necesita, además, actores y aprendiz repartidos en máquinas distintas con corrección V-trace entre ellos. Nada de eso es realista dentro de un único notebook de Colab.*

Lo que sí puedes hacer —y es justo lo que recomienda el consejo de "depura primero en pequeño" antes de dar el salto al paralelismo— es construir el banco de pruebas "Catch" y entrenar sobre él un actor-crítico simple de un solo proceso, con la misma idea que viste en la clase nueve. Sirve para comprobar que la lógica del algoritmo funciona antes de complicar la arquitectura.

Solo necesitas `numpy`, `torch` y `matplotlib`, las tres preinstaladas en Google Colab. Vas a construirlo celda a celda:

1. Importamos librerías
2. Definimos el entorno "Catch"
3. Definimos la red actor-crítico
4. Escribimos la función que juega un episodio completo
5. Escribimos la función que actualiza la red
6. Entrenamos y graficamos la tasa de aciertos

**Ejecuta cada celda en orden.**


In [ ]:
# --- 1. Importamos las librerias ---
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


### El entorno "Catch"

*Guion de clase: dibuja la cuadrícula en la pizarra —diez filas por cinco columnas— antes de mostrar la clase. Marca dónde aparece la pelota y dónde vive la pala, y deja claro que la pala mide dos columnas de ancho.*

La pelota aparece en una columna aleatoria de la fila superior y baja una fila por paso. Controlas una pala de dos píxeles de ancho en la fila inferior, que puedes mover a la izquierda, a la derecha o dejar quieta. Cuando la pelota llega a la fila inferior, recibes +1 si la pala la cubre y -1 si no.


In [ ]:
# --- 2. El entorno "Catch": una pelota cae, una pala debe atraparla ---
FILAS = 10
COLUMNAS = 5
ANCHO_PALA = 2
N_ACCIONES = 3  # 0 = izquierda, 1 = quieta, 2 = derecha


class Catch:
    """Implementacion minima del juego de juguete "Catch"."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.fila_pelota = 0
        self.columna_pelota = np.random.randint(COLUMNAS)
        # Punto de partida aleatorio de la pala, sin salirse del borde
        self.columna_pala = np.random.randint(COLUMNAS - ANCHO_PALA + 1)
        return self._estado()

    def _estado(self):
        """Codifica el estado como una imagen binaria aplanada: un uno
        donde esta la pelota y unos donde ocupa la pala."""
        grid = np.zeros((FILAS, COLUMNAS), dtype=np.float32)
        grid[self.fila_pelota, self.columna_pelota] = 1.0
        grid[FILAS - 1, self.columna_pala:self.columna_pala + ANCHO_PALA] = 1.0
        return grid.flatten()

    def step(self, accion):
        # Mueve la pala; np.clip evita que se salga de la cuadricula
        desplazamiento = {0: -1, 1: 0, 2: 1}[accion]
        self.columna_pala = int(
            np.clip(self.columna_pala + desplazamiento, 0, COLUMNAS - ANCHO_PALA)
        )
        # La pelota cae una fila en cada paso
        self.fila_pelota += 1
        terminado = self.fila_pelota == FILAS - 1
        if terminado:
            atrapada = (
                self.columna_pala
                <= self.columna_pelota
                < self.columna_pala + ANCHO_PALA
            )
            recompensa = 1.0 if atrapada else -1.0
        else:
            recompensa = 0.0
        return self._estado(), recompensa, terminado


### La red actor-crítico

*Guion de clase: señala que hay una sola base compartida y dos cabezas de salida. Pregunta a la clase por qué compartir la base tiene sentido aquí (la política y el valor parten de la misma información sobre el estado).*

Una base compartida procesa el estado; una cabeza produce la política (probabilidad de cada acción) y otra el valor del estado, tal como viste en la clase nueve.


In [ ]:
# --- 3. Red actor-critico: una base compartida y dos cabezas de salida ---
class ActorCritico(nn.Module):
    """Red pequena con una base compartida: una cabeza produce la
    politica (probabilidad de cada accion) y otra el valor del estado."""

    def __init__(self, dim_entrada, n_acciones, dim_oculta=128):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(dim_entrada, dim_oculta),
            nn.ReLU(),
        )
        self.cabeza_politica = nn.Linear(dim_oculta, n_acciones)
        self.cabeza_valor = nn.Linear(dim_oculta, 1)

    def forward(self, x):
        h = self.base(x)
        return self.cabeza_politica(h), self.cabeza_valor(h)


### Hiperparámetros y semillas

Fijamos la semilla aleatoria para que el experimento sea reproducible. `COEF_ENTROPIA` es el consejo práctico "no inicialices la política de forma determinista" hecho hiperparámetro: penaliza que la política se vuelva demasiado segura demasiado pronto.


In [ ]:
# --- 4. Hiperparametros, semillas y objetos del experimento ---
GAMMA = 0.99
TASA_APRENDIZAJE = 1e-3
N_EPISODIOS = 3000
COEF_VALOR = 0.5
COEF_ENTROPIA = 0.01  # fomenta la exploracion: la politica nunca debe
                       # arrancar (ni quedarse) deterministica

torch.manual_seed(0)
np.random.seed(0)

entorno = Catch()
red = ActorCritico(dim_entrada=FILAS * COLUMNAS, n_acciones=N_ACCIONES)
optimizador = torch.optim.Adam(red.parameters(), lr=TASA_APRENDIZAJE)


### Jugar un episodio completo

*Guion de clase: recorre el bucle `while` línea a línea. Insiste en que `distribucion.sample()` es la política muestreando una acción, no eligiendo siempre la mejor.*

Esta función juega a "Catch" de principio a fin con la política actual y guarda, paso a paso, todo lo que necesitas para actualizar la red al terminar: el log-probabilidad de la acción elegida, el valor estimado del estado, la recompensa obtenida y la entropía de la política.


In [ ]:
# --- 5. Un episodio completo con la politica actual ---
def ejecutar_episodio():
    """Juega un episodio completo de Catch y devuelve, paso a paso, el
    log-probabilidad de la accion elegida, el valor estimado del
    estado, la recompensa obtenida y la entropia de la politica."""
    estado = entorno.reset()
    log_probs, valores, recompensas, entropias = [], [], [], []
    terminado = False
    while not terminado:
        entrada = torch.from_numpy(estado)
        logits, valor = red(entrada)
        distribucion = torch.distributions.Categorical(logits=logits)
        accion = distribucion.sample()

        estado, recompensa, terminado = entorno.step(accion.item())

        log_probs.append(distribucion.log_prob(accion))
        entropias.append(distribucion.entropy())
        valores.append(valor.squeeze())
        recompensas.append(recompensa)

    return log_probs, valores, recompensas, entropias


### Actualizar el actor-crítico

*Guion de clase: para en la línea de `ventaja` y pregunta qué pasaría si no restaras `valores.detach()`. Es el mismo concepto de ventaja que ya trabajaste en la clase nueve, ahora dentro de un bucle de entrenamiento completo.*

Calcula el retorno descontado en cada paso, la ventaja frente al valor estimado por el crítico, y actualiza actor y crítico a la vez con una sola pérdida combinada.


In [ ]:
# --- 6. Actualizacion del actor-critico con un episodio completo ---
def actualizar_red(log_probs, valores, recompensas, entropias):
    """Calcula el retorno descontado en cada paso, la ventaja frente al
    valor estimado por el critico, y actualiza actor y critico a la vez."""
    retornos = []
    G = 0.0
    for r in reversed(recompensas):
        G = r + GAMMA * G
        retornos.insert(0, G)
    retornos = torch.tensor(retornos, dtype=torch.float32)

    valores = torch.stack(valores)
    ventaja = retornos - valores.detach()  # sin gradiente hacia el critico aqui

    perdida_actor = -(torch.stack(log_probs) * ventaja).mean()
    perdida_critico = F.mse_loss(valores, retornos)
    perdida_entropia = torch.stack(entropias).mean()

    perdida = (
        perdida_actor
        + COEF_VALOR * perdida_critico
        - COEF_ENTROPIA * perdida_entropia
    )

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


### Entrenamos y medimos la tasa de aciertos

*Guion de clase: antes de ejecutar, pregunta a la clase qué tasa de aciertos esperan al principio. Con tres acciones y dos posiciones de pala en una cuadrícula de cinco columnas, el punto de partida ronda el 40%, cerca del azar. La celda tarda unos segundos: aprovecha para explicar qué está pasando episodio a episodio mientras corre.*

**Ejecuta la celda: entrena 3.000 episodios completos.**


In [ ]:
# --- 7. Bucle de entrenamiento: 3000 episodios completos ---
aciertos = []  # 1 si el episodio termino en captura, 0 si termino en fallo

for episodio in range(N_EPISODIOS):
    log_probs, valores, recompensas, entropias = ejecutar_episodio()
    actualizar_red(log_probs, valores, recompensas, entropias)
    aciertos.append(1 if recompensas[-1] > 0 else 0)

print("Entrenamiento terminado:", N_EPISODIOS, "episodios")


**Ejecuta esta celda para graficar la tasa de aciertos.** Usamos una media móvil de 100 episodios para suavizar el ruido episodio a episodio y dejar ver la tendencia real de aprendizaje.


In [ ]:
# --- 8. Graficamos la tasa de aciertos con una media movil ---
VENTANA = 100
tasa_movil = np.convolve(aciertos, np.ones(VENTANA) / VENTANA, mode="valid")

plt.figure(figsize=(8, 5))
plt.plot(tasa_movil)
plt.xlabel("Episodios")
plt.ylabel(f"Tasa de aciertos (media movil de {VENTANA} episodios)")
plt.title('Actor-critico simple aprendiendo a jugar "Catch"')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()


**Lectura del resultado**

*Guion de clase: para en esta gráfica y pregunta a la clase si la forma de la curva se parece a alguna que ya vieron en clases anteriores (por ejemplo, en la clase nueve con Dyna-Q).*

Verás la tasa de aciertos partir de cerca del azar —en torno al 40%— y subir de forma sostenida hasta acercarse a capturas casi constantes conforme la red aprende a alinear la pala con la columna de la pelota. Si la curva se estanca, prueba a subir `COEF_ENTROPIA` (la política deja de explorar demasiado pronto) o a bajar `TASA_APRENDIZAJE` (las actualizaciones son demasiado bruscas para un problema tan pequeño).


## Puntos clave

- DQN resuelve la correlación de sus datos con una memoria de repetición y una red objetivo, pero eso la ata al aprendizaje fuera de política y desaprovecha la GPU
- A3C sustituye la memoria por el paralelismo: muchos trabajadores con su propia copia del entorno actualizan de forma asíncrona una red compartida, sin bloquearse entre sí
- La diversidad simultánea entre trabajadores rompe la correlación de los datos igual que antes lo hacía la diversidad temporal de la memoria de repetición, sin necesidad de almacenar nada
- El componente que aprende dentro de cada trabajador es un actor-crítico de ventaja: una política normalizada (que suma uno) más un crítico que sirve de referencia
- IMPALA centraliza el aprendizaje en una sola copia y distribuye solo la actuación entre muchos actores, y usa V-trace para corregir el desfase entre la política que generó los datos y la política actual del aprendiz
- IMPALA no solo es más rápido: entrenar varias tareas a la vez con la arquitectura adecuada produce transferencia positiva, mejorando el rendimiento en cada tarea individual
- El juego de juguete "Catch" es el banco de pruebas recomendado para depurar la lógica de un agente antes de complicar la arquitectura con paralelismo
- El diseño de recompensas, el factor de descuento, el tamaño de red y la inicialización no determinista de la política son palancas prácticas tan importantes como el algoritmo elegido

## Antes de la próxima clase

**Abre el notebook de práctica de esta clase y completa los dos ejercicios sobre "Catch".** Vas a programar tú misma o tú mismo la función de paso del entorno desde cero, y vas a medir el efecto real de cambiar el tamaño de la cuadrícula y la velocidad de caída de la pelota.

En la clase 12, la última del curso, ves cómo estas ideas se combinaron —junto con búsqueda en árbol y autojuego— para construir sistemas como AlphaGo, AlphaZero y AlphaStar.
